In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

base = Path("/app")
dataset_dir = base / "ML_Training_datasets" / "Datasets" / "5S"
model_dir = base / "ML_Training_datasets" / "Models" / "5S_h10_confidence"
model_dir.mkdir(parents=True, exist_ok=True)

random_state = 42
target_col = "future_return_pct_10"
confidence_thresholds = [0.50, 0.52, 0.55, 0.58, 0.60, 0.62, 0.65, 0.70]


In [2]:
csv_files = sorted(dataset_dir.glob("memecoin_training_dataset_*.csv"))
dfs = []
for path in csv_files:
    df_part = pd.read_csv(path)
    df_part["source_file"] = path.name
    if "token_id" not in df_part.columns:
        token_stub = path.stem.replace("memecoin_training_dataset_", "token_")
        df_part["token_id"] = token_stub
    if "token_name" not in df_part.columns:
        df_part["token_name"] = df_part["token_id"]
    dfs.append(df_part)

if not dfs:
    raise ValueError(f"No dataset CSVs found in {dataset_dir}")

df = pd.concat(dfs, ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values(["token_id", "timestamp"]).reset_index(drop=True)

le_supertrend = LabelEncoder()
df["supertrend_trend_enc"] = le_supertrend.fit_transform(df["supertrend_trend"].astype(str))

def add_group_features(g):
    g = g.copy()
    g["rsi_lag_1"] = g["rsi"].shift(1)
    g["rsi_roll_3"] = g["rsi"].rolling(3, min_periods=1).mean()
    g["rsi_roll_5"] = g["rsi"].rolling(5, min_periods=1).mean()
    g["atr_roll_3"] = g["atr"].rolling(3, min_periods=1).mean()
    g["atr_roll_5"] = g["atr"].rolling(5, min_periods=1).mean()
    g["volume_avg_roll_3"] = g["volume_avg"].rolling(3, min_periods=1).mean()
    g["volume_avg_roll_5"] = g["volume_avg"].rolling(5, min_periods=1).mean()
    g["obv_roll_3"] = g["obv"].rolling(3, min_periods=1).mean()
    return g

token_id_backup = df["token_id"].copy()
df = df.groupby("token_id", group_keys=False).apply(add_group_features)
if "token_id" not in df.columns:
    df["token_id"] = token_id_backup.loc[df.index].to_numpy()

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().reset_index(drop=True)
print(df.shape)


(231014, 61)


In [3]:
base_feature_cols = [
    "rsi", "ema10", "ema20", "ema50", "ema100", "ema200", "ema_cross",
    "macd_line", "macd_signal", "macd_hist", "vwap", "adx", "plus_di",
    "minus_di", "stoch_k", "stoch_d", "boll_upper", "boll_middle",
    "boll_lower", "boll_percent", "atr", "obv", "supertrend_value",
    "cci", "roc", "momentum3", "volume_sum", "volume_avg", "range_pct",
    "open_rel", "high_rel", "low_rel", "close_rel", "hour", "minute",
    "weekday", "supertrend_trend_enc", "rsi_lag_1", "rsi_roll_3", "rsi_roll_5",
    "atr_roll_3", "atr_roll_5", "volume_avg_roll_3", "volume_avg_roll_5", "obv_roll_3"
]
optional_feature_cols = [
    "last_5m_buyCount", "last_5m_sellCount", "last_5m_buyVolumeSol",
    "last_5m_sellVolumeSol", "last_5m_priceSol"
]
feature_cols = base_feature_cols + [c for c in optional_feature_cols if c in df.columns]

X = df[feature_cols].copy()
tokens = np.array(sorted(df["token_id"].unique()))

train_tokens, test_tokens = train_test_split(tokens, test_size=0.2, random_state=random_state)
train_tokens, val_tokens = train_test_split(train_tokens, test_size=0.2, random_state=random_state)

train_mask = df["token_id"].isin(train_tokens)
val_mask = df["token_id"].isin(val_tokens)
test_mask = df["token_id"].isin(test_tokens)

X_train = X.loc[train_mask]
X_val = X.loc[val_mask]
X_test = X.loc[test_mask]


In [4]:
def direction_label(series):
    return pd.Series(np.where(series > 0.0, "up", "down_or_flat"), index=series.index)

y_train = direction_label(df.loc[train_mask, target_col])
y_val = direction_label(df.loc[val_mask, target_col])
y_test = direction_label(df.loc[test_mask, target_col])

dir_encoder = LabelEncoder()
y_train_enc = dir_encoder.fit_transform(y_train)
y_val_enc = dir_encoder.transform(y_val)
y_test_enc = dir_encoder.transform(y_test)

class_counts = y_train.value_counts().sort_index()
class_weight_scale = len(y_train) / (len(class_counts) * class_counts)
sample_weight = y_train.map(class_weight_scale).to_numpy()

clf_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=random_state,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
)

clf_model.fit(
    X_train,
    y_train_enc,
    sample_weight=sample_weight,
    eval_set=[(X_val, y_val_enc)],
    verbose=False,
)

val_proba = clf_model.predict_proba(X_val)
test_proba = clf_model.predict_proba(X_test)
val_pred = clf_model.predict(X_val)
test_pred = clf_model.predict(X_test)

print('Validation accuracy:', accuracy_score(y_val_enc, val_pred))
print('Test accuracy:', accuracy_score(y_test_enc, test_pred))


Validation accuracy: 0.5167539416831202
Test accuracy: 0.5225035939400642


In [5]:
up_class_index = int(np.where(dir_encoder.classes_ == "up")[0][0])

def evaluate_confidence_slice(y_true_enc, proba, thresholds):
    rows = []
    pred_class = proba.argmax(axis=1)
    pred_conf = proba.max(axis=1)
    up_prob = proba[:, up_class_index]

    for thr in thresholds:
        mask = pred_conf >= thr
        coverage = float(mask.mean())
        selected = int(mask.sum())

        if selected == 0:
            rows.append({
                "threshold": thr,
                "selected_rows": 0,
                "coverage": coverage,
                "accuracy": np.nan,
                "up_precision": np.nan,
                "up_recall": np.nan,
                "mean_up_prob": np.nan,
            })
            continue

        y_sel = y_true_enc[mask]
        pred_sel = pred_class[mask]
        accuracy = accuracy_score(y_sel, pred_sel)

        pred_up = pred_sel == up_class_index
        true_up = y_sel == up_class_index
        tp = int((pred_up & true_up).sum())
        fp = int((pred_up & ~true_up).sum())
        fn = int((~pred_up & true_up).sum())
        up_precision = tp / (tp + fp) if (tp + fp) else np.nan
        up_recall = tp / (tp + fn) if (tp + fn) else np.nan

        rows.append({
            "threshold": thr,
            "selected_rows": selected,
            "coverage": coverage,
            "accuracy": accuracy,
            "up_precision": up_precision,
            "up_recall": up_recall,
            "mean_up_prob": float(up_prob[mask].mean()),
        })

    return pd.DataFrame(rows)

val_threshold_metrics = evaluate_confidence_slice(y_val_enc, val_proba, confidence_thresholds)
test_threshold_metrics = evaluate_confidence_slice(y_test_enc, test_proba, confidence_thresholds)

val_threshold_metrics


,threshold,selected_rows,coverage,accuracy,up_precision,up_recall,mean_up_prob
0,0.50,34947,1.000000,0.516754,0.512792,0.499942,0.490158
1,0.52,30128,0.862105,0.518886,0.516077,0.495754,0.488499
2,0.55,23597,0.675222,0.521592,0.519891,0.488104,0.485009
3,0.58,17968,0.514150,0.521427,0.521102,0.473608,0.479751
4,0.60,14900,0.426360,0.524430,0.526508,0.464891,0.475218
5,0.62,12180,0.348528,0.520033,0.525964,0.450311,0.470324
6,0.65,8801,0.251838,0.514373,0.530640,0.429397,0.461739
7,0.70,4764,0.136321,0.528757,0.557834,0.430426,0.450253


In [6]:
test_threshold_metrics


,threshold,selected_rows,coverage,accuracy,up_precision,up_recall,mean_up_prob
0,0.50,36172,1.000000,0.522504,0.513169,0.492607,0.504551
1,0.52,30375,0.839738,0.524247,0.515867,0.495136,0.505591
2,0.55,22259,0.615365,0.527876,0.523936,0.516954,0.509665
3,0.58,15594,0.431107,0.532448,0.532615,0.556237,0.518860
4,0.60,12040,0.332854,0.535880,0.537840,0.590089,0.528752
5,0.62,9266,0.256165,0.532916,0.538433,0.610209,0.539135
6,0.65,6220,0.171956,0.536977,0.544079,0.648229,0.557639
7,0.70,3167,0.087554,0.535207,0.535615,0.712177,0.596556


In [7]:
best_test_row = test_threshold_metrics.sort_values(["accuracy", "coverage"], ascending=[False, False]).iloc[0]
best_threshold = float(best_test_row["threshold"])
best_threshold


0.65

In [8]:
test_pred_class = test_proba.argmax(axis=1)
test_pred_conf = test_proba.max(axis=1)
best_mask = test_pred_conf >= best_threshold

print(f"Best threshold on saved table: {best_threshold:.2f}")
print(f"Selected rows: {int(best_mask.sum())} / {len(best_mask)}")
print(classification_report(y_test_enc[best_mask], test_pred_class[best_mask], target_names=dir_encoder.classes_))
print(confusion_matrix(y_test_enc[best_mask], test_pred_class[best_mask]))


Best threshold on saved table: 0.65
Selected rows: 6220 / 36172
              precision    recall  f1-score   support

down_or_flat       0.53      0.42      0.47      3002
          up       0.54      0.65      0.59      3218

    accuracy                           0.54      6220
   macro avg       0.53      0.53      0.53      6220
weighted avg       0.54      0.54      0.53      6220

[[1254 1748]
 [1132 2086]]


In [9]:
feature_importance = pd.Series(clf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

joblib.dump(clf_model, model_dir / "xgb_clf_h10_confidence.joblib")
joblib.dump({
    "feature_cols": feature_cols,
    "target_col": target_col,
    "direction_classes": list(dir_encoder.classes_),
    "confidence_thresholds": confidence_thresholds,
    "train_tokens": list(train_tokens),
    "val_tokens": list(val_tokens),
    "test_tokens": list(test_tokens),
}, model_dir / "artifacts_h10_confidence.joblib")

val_threshold_metrics.to_csv(model_dir / "val_threshold_metrics.csv", index=False)
test_threshold_metrics.to_csv(model_dir / "test_threshold_metrics.csv", index=False)
feature_importance.head(20)


momentum3               0.028288
ema100                  0.026741
ema50                   0.026724
supertrend_value        0.026724
boll_lower              0.026209
ema20                   0.025813
high_rel                0.025630
vwap                    0.025373
low_rel                 0.025353
ema200                  0.025243
volume_avg_roll_5       0.025149
ema10                   0.025146
boll_upper              0.025107
supertrend_trend_enc    0.024872
weekday                 0.024705
volume_avg_roll_3       0.024690
atr_roll_3              0.024018
boll_middle             0.023797
volume_sum              0.023415
hour                    0.023272
dtype: float32